In [1]:
import os
import shutil
import subprocess
from kaggle_secrets import UserSecretsClient

GH_USER = "nataliachoszczyk"
GH_REPO = "LLM-Behavior-XAI"
GH_BRANCH = "main"
TARGET_DIR = "/kaggle/working/repo"

secrets = UserSecretsClient()
gh_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{gh_token}@github.com/{GH_USER}/{GH_REPO}.git"

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

subprocess.run(["git", "clone", "--branch", GH_BRANCH, repo_url, TARGET_DIR], check=True)
os.chdir(TARGET_DIR)
print("OK")

Cloning into '/kaggle/working/repo'...


OK


In [2]:
!ls /kaggle/working/repo/data/prompts

test_prompts.csv  train_prompts.csv  valid_prompts.csv


In [3]:
%pip install -q google-genai groq transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
import pandas as pd
from datetime import datetime
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GEMINI_API_KEY = secrets.get_secret("GEMINI_API_KEY")
GROQ_API_KEY   = secrets.get_secret("GROQ_API_KEY")
HF_TOKEN       = secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

In [5]:
# ========== CONFIG ==========

RUN_MODE = "full"  # test/full
TEST_HEAD = 2      # used only when RUN_MODE == "test"
N_RUNS = 1         # independent generations per prompt/model pair

PROMPT_COLUMNS = ["prompt_en", "prompt_pl", "paraphrase_en", "paraphrase_pl"]

CHECKPOINT_EVERY = 10
OUTPUT_PATH = "/kaggle/working/repo/llm_results_20260423_test.csv"
OVERWRITE_OUTPUT_ON_START = True

GENERATION_DEFAULTS = {
    "temperature": 0.3,
    "top_p": 0.9,
    "max_new_tokens": 3072,
    "repetition_penalty": 1.1,
}

In [6]:
PROMPTS_PATH_CANDIDATES = [
    "/kaggle/working/repo/data/prompts/test_prompts.csv"
]

# In test mode prefer the dedicated test file, otherwise prefer full prompts.
if RUN_MODE == "test":
    ordered_candidates = PROMPTS_PATH_CANDIDATES
else:
    ordered_candidates = list(reversed(PROMPTS_PATH_CANDIDATES))

PROMPTS_PATH = next((p for p in ordered_candidates if os.path.exists(p)), None)
if PROMPTS_PATH is None:
    raise FileNotFoundError(
        "Could not find prompts file. Checked: " + ", ".join(ordered_candidates)
    )

# Robust CSV read: sniff delimiter first, then fallback to ';' if needed.
try:
    df_prompts = pd.read_csv(PROMPTS_PATH, sep=None, engine="python", encoding="utf-8")
except pd.errors.ParserError:
    df_prompts = pd.read_csv(PROMPTS_PATH, sep=";", encoding="utf-8")

# Validate columns required by downstream pipeline and output schema.
required_columns = ["prompt_id", "category", *PROMPT_COLUMNS]
missing_columns = [col for col in required_columns if col not in df_prompts.columns]
if missing_columns:
    raise ValueError(
        f"Missing required columns in prompts file: {missing_columns}. "
        f"Available columns: {list(df_prompts.columns)}"
    )

if RUN_MODE == "test":
    df_prompts = df_prompts.head(TEST_HEAD)

df_prompts = df_prompts.copy()
df_prompts["prompt_id"] = df_prompts["prompt_id"].astype(str).str.strip()
df_prompts["category"] = df_prompts["category"].fillna("unknown").astype(str).str.strip()
df_prompts.loc[df_prompts["category"] == "", "category"] = "unknown"

print(f"✅ Loaded {len(df_prompts)} prompts from {PROMPTS_PATH} (mode={RUN_MODE})")
print("✅ Source columns available for output: prompt_id, category")

✅ Loaded 30 prompts from /kaggle/working/repo/data/prompts/test_prompts.csv (mode=full)
✅ Source columns available for output: prompt_id, category


In [7]:
def build_parameters_string(cfg):
    """Create a stable, human-readable parameter summary from config fields."""
    ordered_keys = [
        "temperature",
        "top_p",
        "max_tokens",
        "max_new_tokens",
        "repetition_penalty",
    ]
    parts = []
    for key in ordered_keys:
        value = cfg.get(key, None)
        if value is not None:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


MODELS_CONFIG = {
    "gemini-flash-latest": {
        "provider": "Google Gemini API",
        "model_id": "gemini-flash-latest",
        "temperature": GENERATION_DEFAULTS["temperature"],
        "top_p": GENERATION_DEFAULTS["top_p"],
        "max_tokens": GENERATION_DEFAULTS["max_new_tokens"],
        "type": "api",
    },
    "llama-3.1-8b-groq": {
        "provider": "Groq API",
        "model_id": "llama-3.1-8b-instant",
        "temperature": GENERATION_DEFAULTS["temperature"],
        "top_p": GENERATION_DEFAULTS["top_p"],
        "max_tokens": GENERATION_DEFAULTS["max_new_tokens"],
        "type": "api",
    },
    "mistral-7b-hf": {
        "provider": "HuggingFace (local GPU)",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "temperature": GENERATION_DEFAULTS["temperature"],
        "top_p": GENERATION_DEFAULTS["top_p"],
        "max_new_tokens": GENERATION_DEFAULTS["max_new_tokens"],
        "repetition_penalty": GENERATION_DEFAULTS["repetition_penalty"],
        "type": "local",
    },
    "phi-3-mini-hf": {
        "provider": "HuggingFace (local GPU)",
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "temperature": GENERATION_DEFAULTS["temperature"],
        "top_p": GENERATION_DEFAULTS["top_p"],
        "max_new_tokens": GENERATION_DEFAULTS["max_new_tokens"],
        "repetition_penalty": GENERATION_DEFAULTS["repetition_penalty"],
        "type": "local",
    },
}

for cfg in MODELS_CONFIG.values():
    cfg["parameters"] = build_parameters_string(cfg)

print("✅ Model configuration loaded")
for name, cfg in MODELS_CONFIG.items():
    print(f"{name} ({cfg['provider']}) | {cfg['parameters']}")

✅ Model configuration loaded
gemini-flash-latest (Google Gemini API) | temperature=0.3, top_p=0.9, max_tokens=3072
llama-3.1-8b-groq (Groq API) | temperature=0.3, top_p=0.9, max_tokens=3072
mistral-7b-hf (HuggingFace (local GPU)) | temperature=0.3, top_p=0.9, max_new_tokens=3072, repetition_penalty=1.1
phi-3-mini-hf (HuggingFace (local GPU)) | temperature=0.3, top_p=0.9, max_new_tokens=3072, repetition_penalty=1.1


In [8]:
from google import genai
from google.genai import types
from groq import Groq

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Gemini initialized")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq initialized")

✅ Gemini initialized
✅ Groq initialized


In [9]:
import os
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
local_models = {}


def load_local_model(model_id, model_key):
    """Load a Hugging Face model on GPU with 4-bit quantization."""
    print(f"⏳ Downloading model: {model_id} ...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    use_remote_code = model_key != "phi-3-mini-hf"

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=HF_TOKEN,
        trust_remote_code=use_remote_code,
    )

    config = AutoConfig.from_pretrained(
        model_id,
        token=HF_TOKEN,
        trust_remote_code=use_remote_code,
    )
    if model_key == "phi-3-mini-hf":
        rope_parameters = getattr(config, "rope_parameters", None)
        if not isinstance(rope_parameters, dict):
            rope_parameters = {}
        rope_parameters["rope_type"] = "default"
        config.rope_parameters = rope_parameters
        config._attn_implementation = "eager"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        config=config,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=use_remote_code,
    )
    model.eval()

    print(f"✅ Model {model_key} loaded!")
    return {"model": model, "tokenizer": tokenizer}


local_models["mistral-7b-hf"] = load_local_model(
    MODELS_CONFIG["mistral-7b-hf"]["model_id"], "mistral-7b-hf"
)
local_models["phi-3-mini-hf"] = load_local_model(
    MODELS_CONFIG["phi-3-mini-hf"]["model_id"], "phi-3-mini-hf"
)

print("\n✅ All local models loaded!")

⏳ Downloading model: mistralai/Mistral-7B-Instruct-v0.3 ...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model phi-3-mini-hf loaded!

✅ All local models loaded!


In [10]:
import math


def summarize_logprobs(token_logprobs):
    """Aggregate per-token log-probabilities into stable summary metrics."""
    if not token_logprobs:
        return {
            "logprob_available": False,
            "sum_logprob": None,
            "avg_logprob": None,
            "generated_tokens": 0,
            "perplexity": None,
        }

    total = float(sum(token_logprobs))
    n_tokens = int(len(token_logprobs))
    avg = total / n_tokens
    ppl = float(math.exp(-avg))

    return {
        "logprob_available": True,
        "sum_logprob": round(total, 6),
        "avg_logprob": round(avg, 6),
        "generated_tokens": n_tokens,
        "perplexity": round(ppl, 6),
    }


def query_gemini(prompt_text, config):
    """Query Gemini via Google Gen AI SDK."""
    generation_config = types.GenerateContentConfig(
        temperature=config["temperature"],
        max_output_tokens=config["max_tokens"],
        top_p=config.get("top_p", 0.95),
    )

    configured = config.get("model_id", "gemini-flash-latest")
    if configured.startswith("models/"):
        configured_short = configured.replace("models/", "", 1)
    else:
        configured_short = configured

    model_candidates = [
        configured,
        configured_short,
        f"models/{configured_short}",
        "gemini-flash-latest",
        "models/gemini-flash-latest",
        "gemini-2.0-flash",
        "models/gemini-2.0-flash",
    ]
    last_error = None

    for model_id in dict.fromkeys(model_candidates):
        try:
            response = gemini_client.models.generate_content(
                model=model_id,
                contents=prompt_text,
                config=generation_config,
            )
            text = (response.text or "").strip()
            if not text:
                return None, "Empty Gemini response", summarize_logprobs([])
            
            finish = getattr(response.candidates[0], "finish_reason", None)
            if str(finish) not in ("STOP", "FinishReason.STOP", "1"):
                print(f"  ⚠️ finish_reason={finish} (response may have been truncated)")

            return text, None, summarize_logprobs([])
        except Exception as e:
            err = str(e)
            last_error = err
            if "NOT_FOUND" not in err and "not found" not in err.lower():
                return None, err, summarize_logprobs([])

    return None, last_error or "Gemini request failed", summarize_logprobs([])


def query_groq(prompt_text, config):
    """Query a model via Groq API with optional logprobs."""
    try:
        request = {
            "model": config["model_id"],
            "messages": [{"role": "user", "content": prompt_text}],
            "temperature": config["temperature"],
            "top_p": config.get("top_p", 1.0),
            "max_tokens": config["max_tokens"],
            "logprobs": True,
            "top_logprobs": 1,
        }

        try:
            completion = groq_client.chat.completions.create(**request)
        except Exception as e:
            if "logprob" not in str(e).lower():
                raise
            request.pop("logprobs", None)
            request.pop("top_logprobs", None)
            completion = groq_client.chat.completions.create(**request)

        choice = completion.choices[0]
        response_text = (choice.message.content or "").strip()

        token_logprobs = []
        choice_logprobs = getattr(choice, "logprobs", None)
        if choice_logprobs is not None:
            content_items = getattr(choice_logprobs, "content", None)
            if content_items is not None:
                for item in content_items:
                    lp = getattr(item, "logprob", None)
                    if lp is not None:
                        token_logprobs.append(float(lp))

        return response_text, None, summarize_logprobs(token_logprobs)
    except Exception as e:
        return None, str(e), summarize_logprobs([])


def query_local_hf(prompt_text, model_key, config):
    """Query a local Hugging Face model and return token-level logprobs."""
    try:
        local_bundle = local_models[model_key]
        model = local_bundle["model"]
        tokenizer = local_bundle["tokenizer"]

        messages = [{"role": "user", "content": prompt_text}]
        prompt_for_model = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(prompt_for_model, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=config["max_new_tokens"],
                temperature=config["temperature"],
                top_p=config.get("top_p", 1.0),
                repetition_penalty=config.get("repetition_penalty", 1.0),
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_scores=True,
            )

        prompt_len = int(inputs["input_ids"].shape[-1])
        generated_ids = generated.sequences[0][prompt_len:]

        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        token_logprobs = []
        for step, step_scores in enumerate(generated.scores[: len(generated_ids)]):
            token_id = int(generated_ids[step].item())
            log_probs = torch.log_softmax(step_scores[0], dim=-1)
            token_logprobs.append(float(log_probs[token_id].item()))

        return response_text, None, summarize_logprobs(token_logprobs)
    except Exception as e:
        return None, str(e), summarize_logprobs([])


def query_model(model_key, prompt_text):
    """Dispatch query to the correct model handler."""
    config = MODELS_CONFIG[model_key]

    start_time = time.time()

    if config["provider"] == "Google Gemini API" or model_key.startswith("gemini"):
        response, error, logprob_stats = query_gemini(prompt_text, config)
    elif model_key == "llama-3.1-8b-groq":
        response, error, logprob_stats = query_groq(prompt_text, config)
    elif model_key in local_models:
        response, error, logprob_stats = query_local_hf(prompt_text, model_key, config)
    else:
        response, error, logprob_stats = None, f"Unknown model: {model_key}", summarize_logprobs([])

    elapsed = round(time.time() - start_time, 2)

    return response, error, elapsed, logprob_stats


print("✅ Query functions defined")

✅ Query functions defined


In [11]:
results = []
checkpoint_buffer = []

if OVERWRITE_OUTPUT_ON_START and os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)


def flush_checkpoint(rows, output_path):
    """Append buffered rows to CSV checkpoint file."""
    if not rows:
        return

    chunk_df = pd.DataFrame(rows)
    write_header = not os.path.exists(output_path)
    chunk_df.to_csv(
        output_path,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig",
    )


total_calls = len(df_prompts) * len(MODELS_CONFIG) * len(PROMPT_COLUMNS) * N_RUNS
call_count = 0

print(
    f"Pipeline start | {len(df_prompts)} prompts × {len(MODELS_CONFIG)} models × "
    f"{len(PROMPT_COLUMNS)} variants × {N_RUNS} runs = {total_calls} calls\n"
)

for run_id in range(1, N_RUNS + 1):
    print(f"\n🔁 Run {run_id}/{N_RUNS}")

    for _, row in df_prompts.iterrows():
        prompt_id = str(row["prompt_id"]).strip()
        category = str(row["category"]).strip() or "unknown"

        for prompt_col in PROMPT_COLUMNS:
            prompt_text = row[prompt_col]
            lang = "en" if "_en" in prompt_col else "pl"
            is_paraphrase = "paraphrase" in prompt_col

            for model_key in MODELS_CONFIG:
                call_count += 1
                cfg = MODELS_CONFIG[model_key]

                print(
                    f"[{call_count}/{total_calls}] run={run_id} "
                    f"prompt_id={prompt_id} | {prompt_col} | {model_key} ...",
                    end=" ",
                )

                response, error, elapsed, logprob_stats = query_model(model_key, prompt_text)

                row_out = {
                    "run_id": run_id,
                    "prompt_id": prompt_id,
                    "category": category,
                    "prompt_column": prompt_col,
                    "language": lang,
                    "is_paraphrase": is_paraphrase,
                    "prompt_text": prompt_text,
                    "model_key": model_key,
                    "model_id": cfg["model_id"],
                    "provider": cfg["provider"],
                    "model_parameters": cfg["parameters"],
                    "temperature": cfg["temperature"],
                    "top_p": cfg.get("top_p", None),
                    "max_tokens": cfg.get("max_tokens", None),
                    "max_new_tokens": cfg.get("max_new_tokens", None),
                    "repetition_penalty": cfg.get("repetition_penalty", None),
                    "response": response if response else "",
                    "error": error if error else "",
                    "response_length": len(response) if response else 0,
                    "elapsed_seconds": elapsed,
                    "logprob_available": logprob_stats.get("logprob_available", False),
                    "sum_logprob": logprob_stats.get("sum_logprob", None),
                    "avg_logprob": logprob_stats.get("avg_logprob", None),
                    "generated_tokens": logprob_stats.get("generated_tokens", 0),
                    "perplexity": logprob_stats.get("perplexity", None),
                    "timestamp": datetime.now().isoformat(),
                }

                results.append(row_out)
                checkpoint_buffer.append(row_out)

                if response:
                    base = f"✅ ({elapsed}s, {len(response)} chars)"
                    if logprob_stats.get("logprob_available"):
                        base += f" | avg_logprob={logprob_stats.get('avg_logprob')}"
                    print(base)
                else:
                    print(f"❌ {error}")

                if len(checkpoint_buffer) >= CHECKPOINT_EVERY:
                    flush_checkpoint(checkpoint_buffer, OUTPUT_PATH)
                    print(f"💾 Checkpoint saved: +{len(checkpoint_buffer)} rows")
                    checkpoint_buffer = []

                # Short delay to reduce API rate-limit risk
                if cfg["type"] == "api":
                    time.sleep(0.5)

# Final checkpoint flush
flush_checkpoint(checkpoint_buffer, OUTPUT_PATH)
if checkpoint_buffer:
    print(f"💾 Final checkpoint saved: +{len(checkpoint_buffer)} rows")

print(f"\n✅ Pipeline complete! Collected {len(results)} results.")
print(f"✅ Incremental CSV saved to: {OUTPUT_PATH}")

Pipeline start | 30 prompts × 4 models × 4 variants × 1 runs = 480 calls


🔁 Run 1/1
[1/480] run=1 prompt_id=EXP_T01 | prompt_en | gemini-flash-latest ... ✅ (8.65s, 2849 chars)
[2/480] run=1 prompt_id=EXP_T01 | prompt_en | llama-3.1-8b-groq ... ✅ (1.53s, 2110 chars)
[3/480] run=1 prompt_id=EXP_T01 | prompt_en | mistral-7b-hf ... ✅ (37.85s, 1907 chars) | avg_logprob=-0.153392
[4/480] run=1 prompt_id=EXP_T01 | prompt_en | phi-3-mini-hf ... ✅ (22.37s, 1464 chars) | avg_logprob=-0.318046
[5/480] run=1 prompt_id=EXP_T01 | prompt_pl | gemini-flash-latest ... ✅ (10.24s, 3133 chars)
[6/480] run=1 prompt_id=EXP_T01 | prompt_pl | llama-3.1-8b-groq ... ✅ (2.31s, 1789 chars)
[7/480] run=1 prompt_id=EXP_T01 | prompt_pl | mistral-7b-hf ... ✅ (74.04s, 2008 chars) | avg_logprob=-0.107822
[8/480] run=1 prompt_id=EXP_T01 | prompt_pl | phi-3-mini-hf ... ✅ (46.05s, 1732 chars) | avg_logprob=-0.219315
[9/480] run=1 prompt_id=EXP_T01 | paraphrase_en | gemini-flash-latest ... ✅ (9.54s, 3670 chars)
[10/480] r